## GenAI Tracing with MLflow

### Installing Utilities and Libraries

In [ ]:
%pip install -qq --upgrade "mlflow[databricks]>=3.1.0" openai databricks-sdk==0.77.0

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Creating the OpenAI Client

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
    base_url = "YOUR_DATABRICKS_WORKSPACE_HOSTNAME/serving-endpoints"
)

### Trace your Application

In [ ]:
import mlflow
import os

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Trace-LLM-Application")

# Use the trace decorator to capture the application's entry point
@mlflow.trace
def my_app(input: str):
    # This call is automatically instrumented by `mlflow.openai.autolog()`
    # OpenAI Request
    completion = client.chat.completions.create(
        model="databricks-claude-sonnet-4-5",
        messages=[
            {
                "role":"system",
                "content": [
                    {
                        "type": "text", "text": "You are Batman, the protector of Gotham City"
                    }
                ]

            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text", "text": input
                    }
                ]
            }
        ]
    )

    # printing the response
    return completion.choices[0].message.content

### Invoke your Application Trace

In [ ]:
result = my_app(input = "How is Gotham City doing today?")
print(result)